In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import solve_ivp
import ipywidgets
import plotly.graph_objects as go
import rasterio
from matplotlib.colors import LightSource
from ipywidgets import interact, FloatSlider, IntSlider
import plotly.io as pio
from scipy.spatial import Delaunay
import pyvista as pv
import rasterio
from scipy.interpolate import RegularGridInterpolator
from plotly.subplots import make_subplots
from rasterio.enums import Resampling
from rasterio.windows import Window
pio.renderers.default = 'browser'
from utils import *

In [2]:
file_path = 'LDEM_20M.tif' # Loads .tif
dimension_scene, _ = read_tif(file_path) # Gets dimension of the scene
visualize_tif(file_path, np.floor(dimension_scene / 4)) # Shows half the scene (cause it is big)


--- PROPRIETÀ BASE DEL RASTER ---
Larghezza (Colonne):   7584 pixel
Altezza (Righe):       7584 pixel
Risoluzione pixel (X, Y): (20.0, 20.0) metri
Bordi della mappa (Bounds):
BoundingBox(left=-75840.0, bottom=-75840.0, right=75840.0, top=75840.0)

--- 🏔️ ANALISI STATISTICA ALTEZZE LUNARI ---
Punto più profondo (Crateri): -8758 metri
Punto più elevato (Picchi):     4744 metri
Dislivello totale nella scena:  13502 metri
Elevazione media del terreno:   -1959.06 metri
Variabilità del terreno (Std):  2261.61 metri
--- 🔍 VISUALIZZAZIONE FINESTRA ---
Risoluzione renderizzata a schermo: 1896.0x1896.0 pixel.


In [3]:
file_cut = 'LDEM_20M_clip.tif' # Creates the cut file's path
cut_pixels = 500 # Choose the square, starting from the center, that is going to be cut from the whole scene

write_and_save_tif(file_path, file_cut, cut_pixels, 0, 0) # Saves the cut scene

dimension_scene_cut, _ = read_tif(file_cut) # Read the cut scene
visualize_tif(file_cut, dimension_scene_cut) # Viualizes it


✅ File salvato con successo: 'LDEM_20M_clip.tif'
Coordinate inserite: X = 0 km, Y = 0 km
Convertite in indici pixel centrali: (3792, 3792)
Finestra applicata (500x500 px): Colonne [3542:4042], Righe [3542:4042]
--- PROPRIETÀ BASE DEL RASTER ---
Larghezza (Colonne):   500 pixel
Altezza (Righe):       500 pixel
Risoluzione pixel (X, Y): (20.0, 20.0) metri
Bordi della mappa (Bounds):
BoundingBox(left=-5000.0, bottom=-5000.0, right=5000.0, top=5000.0)

--- 🏔️ ANALISI STATISTICA ALTEZZE LUNARI ---
Punto più profondo (Crateri): -5120 metri
Punto più elevato (Picchi):     3337 metri
Dislivello totale nella scena:  8457 metri
Elevazione media del terreno:   833.66 metri
Variabilità del terreno (Std):  1710.30 metri
--- 🔍 VISUALIZZAZIONE FINESTRA ---
Risoluzione renderizzata a schermo: 500x500 pixel.


In [4]:
passo = cut_pixels 
X, Y, Z, triangoli, Xe, Ye, Ze, normali, baricentri, Xg, Yg, Zg = tif2mesh(file_cut, np.floor(dimension_scene_cut*3), passo) # Creates the mesh of triangles from the starting .tif

file_cut_up = 'vesuvio_clip_up.tif' # Creates file's path for the cut and upsampled scene 
nuova_risoluzione_metri = 10
passo_up = cut_pixels * 2
upsampling_geotiff(file_cut, file_cut_up, nuova_risoluzione_metri, metodo_interpolazione='lanczos') # Upsamples the cut file to a new resolution
X_up, Y_up, Z_up, triangoli_up, Xe_up, Ye_up, Ze_up, normali_up, baricentri_up, Xg_up, Yg_up, Zg_up  = tif2mesh(file_cut_up, np.floor(dimension_scene_cut*3), passo_up) # Creates the mesh of triangles from the starting .tif

show_normals = False
show_triangles = False

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=(
        f"Mesh Originale (20m) - Triangoli: {len(triangoli):,}", 
        f"Mesh Interpolata {nuova_risoluzione_metri} m - Triangoli: {len(triangoli_up):,}"
    )
)
fig.add_trace(go.Mesh3d(
    x=X, y=Y, z=Z, i=triangoli[:, 0], j=triangoli[:, 1], k=triangoli[:, 2],
    intensity=Z, colorscale='Earth_r', showscale=False, opacity=0.85, name="Orig 20m"
), row=1, col=1)

if show_triangles:
    fig.add_trace(go.Scatter3d(
        x=Xe, y=Ye, z=Ze, mode='lines',
        line=dict(color='rgb(30,30,30)', width=1), showlegend=False
    ), row=1, col=1)

if show_normals:
    fig.add_trace(go.Scatter3d(
        x=Xg, y=Yg, z=Zg, mode='lines',
        line=dict(color='rgb(255, 0, 0)', width=3), name="Vettori Normali", showlegend=False
    ), row=1, col=1)

fig.add_trace(go.Mesh3d(
    x=X_up, y=Y_up, z=Z_up, i=triangoli_up[:, 0], j=triangoli_up[:, 1], k=triangoli_up[:, 2],
    intensity=Z_up, colorscale='Earth_r', showscale=True, opacity=0.85,
    colorbar=dict(title="Quota (m)", x=1.05), name="Interp 10m"
), row=1, col=2)

if show_triangles:
    fig.add_trace(go.Scatter3d(
        x=Xe_up, y=Ye_up, z=Ze_up, mode='lines',
        line=dict(color='rgb(50,50,50)', width=0.8), showlegend=False
    ), row=1, col=2)

if show_normals:
    fig.add_trace(go.Scatter3d(
        x=Xg_up, y=Yg_up, z=Zg_up, mode='lines',
        line=dict(color='rgb(255, 0, 0)', width=3), name="Vettori Normali", showlegend=False
    ), row=1, col=2)

scene_config = dict(
    xaxis_title="X (Metri)",
    yaxis_title="Y (Metri)",
    zaxis_title="Z (Metri)",
    aspectmode='data' 
    # aspectmode='manual',
    # aspectratio=dict(x=1, y=1, z=0.4)
)

fig.update_layout(
    scene=scene_config,   # Configura la vista di sinistra (scena 1)
    scene2=scene_config,  # Configura la vista di destra (scena 2)
    margin=dict(l=0, r=50, b=0, t=110),
    width=1150,           # Allargato leggermente per fare spazio alla barra dei colori
    height=750
)
fig.show()


--- 🌋 ANALISI GEOMORFOLOGICA DEL TERRENO ---
Pendenza Massima Rilevata: 82.0°
Pendenza Media Rilevata:  38.6°
Quota Media di Riferimento: 837.0 m

Ripartizione della Mesh:
 🟩 Pianura in Depressione (Bacino/Cratere): 0.4%
 🟨 Pianura Elevata (Altopiano):             0.4%
 🟧 Pendii e Raccordi Dolci:                  10.0%
 🟥 Versanti Ripidi (Montagna/Bordi Cratere): 89.2%
--- 🔄 AVVIO RICAMPIONAMENTO ---
Risoluzione originale: 20.0 m/px -> Dimensione: 500x500
Nuova risoluzione:     10 m/px -> Dimensione: 1000x1000
Metodo utilizzato:     lanczos
🎉 Salvataggio completato con successo: 'vesuvio_clip_up.tif'

--- 🌋 ANALISI GEOMORFOLOGICA DEL TERRENO ---
Pendenza Massima Rilevata: 82.8°
Pendenza Media Rilevata:  38.5°
Quota Media di Riferimento: 835.4 m

Ripartizione della Mesh:
 🟩 Pianura in Depressione (Bacino/Cratere): 0.2%
 🟨 Pianura Elevata (Altopiano):             0.2%
 🟧 Pendii e Raccordi Dolci:                  11.2%
 🟥 Versanti Ripidi (Montagna/Bordi Cratere): 88.4%
